In [1]:
import pandas as pd
import numpy as np
import warnings, time
warnings.filterwarnings('ignore')

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score

try:
    import pygeohash as pgh
    GEO_OK = True
except ImportError:
    GEO_OK = False

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

SEED   = 42
NFOLDS = 5
np.random.seed(SEED)
t0 = time.time()

In [5]:
!pip install lightgbm xgboost catboost pygeohash --quiet


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# ─────────────────────────── 1. LOAD DATA ───────────────────────
print("=" * 60)
print("STEP 1: Loading data")
print("=" * 60)
train = pd.read_csv("train.csv")
test  = pd.read_csv("test.csv")
print(f"  Train: {train.shape}  |  Test: {test.shape}")
print(f"  Train days: {sorted(train['day'].unique())}")
print(f"  Test  days: {sorted(test['day'].unique())}")
print(f"  Demand range: {train['demand'].min():.4f} – {train['demand'].max():.4f}")
print(f"  Demand skew : {train['demand'].skew():.3f}")

STEP 1: Loading data
  Train: (77299, 11)  |  Test: (41778, 10)
  Train days: [np.int64(48), np.int64(49)]
  Test  days: [np.int64(49)]
  Demand range: 0.0000 – 1.0000
  Demand skew : 3.729


In [3]:
# ------------------parse timestamp-------------
def parse_time(df):
    df = df.copy()
    parts = df['timestamp'].str.split(':', expand=True).astype(int)
    df['hour']   = parts[0]
    df['minute'] = parts[1]
    df['hhmm']   = df['hour'] * 60 + df['minute']   # 0–1425 (every 15 min)
    return df

train = parse_time(train)
test  = parse_time(test)
print(f"\n  Timestamp slots in train: {train['timestamp'].nunique()}")
print(f"  Timestamp slots in test : {test['timestamp'].nunique()}")


  Timestamp slots in train: 96
  Timestamp slots in test : 47


In [8]:
# ─────────────────────────── 3. LAG FEATURE: Day-48 demand → Day-49 predictor
#    Lag1 correlation with target = 0.97 — biggest single feature! ─────────────
print("\n" + "=" * 60)
print("STEP 3: Building Day-48 lag features")
print("=" * 60)

day48 = train[train['day'] == 48][['geohash','hhmm','demand']].copy()
day48.rename(columns={'demand': 'demand_d48'}, inplace=True)

# Lag -1 slot (15 min before) on day 48
day48_sorted = day48.sort_values(['geohash','hhmm'])
day48_sorted['demand_d48_lag1']   = day48_sorted.groupby('geohash')['demand_d48'].shift(1)
day48_sorted['demand_d48_lag2']   = day48_sorted.groupby('geohash')['demand_d48'].shift(2)
day48_sorted['demand_d48_lead1']  = day48_sorted.groupby('geohash')['demand_d48'].shift(-1)
day48_sorted['demand_d48_roll3']  = day48_sorted.groupby('geohash')['demand_d48'].transform(
                                        lambda x: x.rolling(3, min_periods=1).mean())
day48_sorted['demand_d48_roll6']  = day48_sorted.groupby('geohash')['demand_d48'].transform(
                                        lambda x: x.rolling(6, min_periods=1).mean())

lag_cols = ['geohash','hhmm','demand_d48','demand_d48_lag1','demand_d48_lag2',
            'demand_d48_lead1','demand_d48_roll3','demand_d48_roll6']
day48_lags = day48_sorted[lag_cols].drop_duplicates(['geohash','hhmm'])

# Merge onto train and test
train = train.merge(day48_lags, on=['geohash','hhmm'], how='left')
test  = test.merge(day48_lags,  on=['geohash','hhmm'], how='left')

# Fill missing lag values with geohash mean from day 48
gh_d48_mean = day48.groupby('geohash')['demand_d48'].mean().rename('_gh_d48_mean')
train = train.join(gh_d48_mean, on='geohash')
test  = test.join(gh_d48_mean,  on='geohash')
for col in ['demand_d48','demand_d48_lag1','demand_d48_lag2',
            'demand_d48_lead1','demand_d48_roll3','demand_d48_roll6']:
    train[col].fillna(train['_gh_d48_mean'], inplace=True)
    test[col].fillna(test['_gh_d48_mean'], inplace=True)
train.drop('_gh_d48_mean', axis=1, inplace=True)
test.drop('_gh_d48_mean',  axis=1, inplace=True)

lag_match_test = test['demand_d48'].notna().mean()
print(f"  Day-48 lag coverage in test: {lag_match_test:.1%}")
print(f"  Lag-1 corr with demand (train): {train['demand_d48'].corr(train['demand']):.4f}")


 STEP 3: Feature Engineering
  After base FE  → train: (77299, 55), test: (41778, 54)


In [4]:
# ─────────────────────────── 4. LOG-TRANSFORM TARGET
#    Demand skew = 3.7 → log1p makes tree splits more effective ──────
train['log_demand'] = np.log1p(train['demand'])
TARGET = 'log_demand'

In [5]:
# ─────────────────────────── 5. FEATURE ENGINEERING  ──────────────
print("\n" + "=" * 60)
print("STEP 5: Feature Engineering")
print("=" * 60)

def decode_geo(gh):
    try:
        return pgh.decode(str(gh))
    except:
        return (np.nan, np.nan)

def engineer(df, geo_map=None, encoders=None):
    df = df.copy()

    # ── A. GEOHASH spatial hierarchy ───────────────────────────────
    if GEO_OK:
        if geo_map is None:
            geo_map = {gh: decode_geo(gh) for gh in df['geohash'].dropna().unique()}
        df['lat'] = df['geohash'].map(lambda x: geo_map.get(x,(np.nan,np.nan))[0])
        df['lon'] = df['geohash'].map(lambda x: geo_map.get(x,(np.nan,np.nan))[1])
    else:
        df['lat'] = np.nan
        df['lon'] = np.nan

    df['geo3'] = df['geohash'].astype(str).str[:3]
    df['geo4'] = df['geohash'].astype(str).str[:4]
    df['geo5'] = df['geohash'].astype(str).str[:5]

    # ── B. TIME features (timestamp = "H:M" already parsed) ────────
    df['is_peak_am']  = df['hour'].between(7, 9).astype(int)
    df['is_peak_pm']  = df['hour'].between(17,19).astype(int)
    df['is_night']    = (~df['hour'].between(6, 22)).astype(int)
    df['is_midday']   = df['hour'].between(11,13).astype(int)
    df['hour_sin']    = np.sin(2 * np.pi * df['hour'] / 24)
    df['hour_cos']    = np.cos(2 * np.pi * df['hour'] / 24)
    df['hhmm_sin']    = np.sin(2 * np.pi * df['hhmm'] / 1440)
    df['hhmm_cos']    = np.cos(2 * np.pi * df['hhmm'] / 1440)

    # ── C. DAY feature — integer (48 or 49), NOT day name! ─────────
    # v1 bug: tried to map 'Monday'… to integer → all became -1
    # Fix: use day directly as numeric; add relative flag
    df['day_rel'] = df['day'] - df['day'].min()   # 0 or 1

    # ── D. ROAD features — BIGGEST predictors ──────────────────────
    # RoadType: Highway >> Street >> Residential
    df['NumberofLanes'] = pd.to_numeric(df['NumberofLanes'], errors='coerce').fillna(1)
    df['is_highway']    = (df['RoadType'] == 'Highway').astype(int)
    df['is_street']     = (df['RoadType'] == 'Street').astype(int)
    df['is_residential']= (df['RoadType'] == 'Residential').astype(int)
    df['high_capacity'] = (df['NumberofLanes'] >= 4).astype(int)

    # RoadType × Lanes interaction — critical combo
    df['road_lanes']    = df['RoadType'].astype(str) + '_L' + df['NumberofLanes'].astype(str)

    for col in ['LargeVehicles','Landmarks']:
        if col in df.columns:
            df[col] = (df[col].astype(str).str.strip().str.lower()
                       .isin(['allowed','yes','1','true'])).astype(int)

    # ── E. WEATHER / TEMPERATURE ────────────────────────────────────
    df['Temperature']  = pd.to_numeric(df['Temperature'], errors='coerce')
    df['temp_missing'] = df['Temperature'].isnull().astype(int)
    df['Temperature'].fillna(df['Temperature'].median(), inplace=True)

    # ── F. INTERACTIONS ─────────────────────────────────────────────
    df['geo4_hour']    = df['geo4'] + '_h' + df['hour'].astype(str)
    df['geo4_hhmm']    = df['geo4'] + '_t' + df['hhmm'].astype(str)   # finer: per 15-min slot
    df['geo4_day']     = df['geo4'] + '_d' + df['day'].astype(str)
    df['road_hour']    = df['RoadType'].astype(str) + '_h' + df['hour'].astype(str)
    df['road_hhmm']    = df['RoadType'].astype(str) + '_t' + df['hhmm'].astype(str)

    # ── G. LABEL ENCODE categorical cols ───────────────────────────
    cat_cols = ['RoadType','Weather',
                'geo3','geo4','geo5','geohash',
                'geo4_hour','geo4_hhmm','geo4_day',
                'road_hour','road_hhmm','road_lanes']
    if encoders is None:
        encoders = {}
    for col in cat_cols:
        if col not in df.columns: continue
        df[col] = df[col].astype(str).fillna('MISSING')
        if col not in encoders:
            le = LabelEncoder()
            df[col+'_enc'] = le.fit_transform(df[col])
            encoders[col]  = le
        else:
            le    = encoders[col]
            known = set(le.classes_)
            df[col] = df[col].map(lambda x: x if x in known else 'MISSING')
            if 'MISSING' not in known:
                le.classes_ = np.append(le.classes_, 'MISSING')
            df[col+'_enc'] = le.transform(df[col])

    return df, geo_map, encoders

train_eng, geo_map, encoders = engineer(train)
test_eng,  _,       _        = engineer(test, geo_map=geo_map, encoders=encoders)
print(f"  After base FE → train: {train_eng.shape}, test: {test_eng.shape}")


STEP 5: Feature Engineering
  After base FE → train: (77299, 52), test: (41778, 50)


In [6]:
# ─────────────────────────── 6. TARGET AGGREGATION FEATURES ───────────────────
print("\n" + "=" * 60)
print("STEP 6: Target Aggregation Features")
print("=" * 60)

def target_agg(tr, te, grp_col, tgt='demand', prefix=None):
    prefix = prefix or grp_col
    stats = tr.groupby(grp_col)[tgt].agg(
        **{f'{prefix}_mean':   'mean',
           f'{prefix}_median': 'median',
           f'{prefix}_std':    'std',
           f'{prefix}_max':    'max'}
    ).reset_index()
    tr2 = tr.merge(stats, on=grp_col, how='left')
    te2 = te.merge(stats, on=grp_col, how='left')
    for c in stats.columns[1:]:
        fv = tr[tgt].mean() if 'mean' in c else \
             tr[tgt].median() if 'median' in c else \
             tr[tgt].std() if 'std' in c else tr[tgt].max()
        tr2[c].fillna(fv, inplace=True)
        te2[c].fillna(fv, inplace=True)
    return tr2, te2

AGG_GROUPS = [
    ('geohash',       'demand', 'gh'),          # per exact location
    ('geo3',          'demand', 'geo3'),
    ('geo4',          'demand', 'geo4'),
    ('geo5',          'demand', 'geo5'),
    ('hhmm',          'demand', 'hhmm'),         # per 15-min slot
    ('hour',          'demand', 'hour'),
    ('RoadType_enc',  'demand', 'road'),
    ('road_lanes',    'demand', 'road_lanes'),    # road × lanes combo
    ('geo4_hour',     'demand', 'geo4_hour'),
    ('geo4_hhmm',     'demand', 'geo4_hhmm'),    # location × 15-min
    ('geo4_day',      'demand', 'geo4_day'),
    ('road_hour',     'demand', 'road_hour'),
    ('road_hhmm',     'demand', 'road_hhmm'),
]

for grp_col, tgt, prefix in AGG_GROUPS:
    if grp_col in train_eng.columns:
        train_eng, test_eng = target_agg(train_eng, test_eng, grp_col, tgt, prefix)
        print(f"  ✓ {grp_col}")

print(f"\n  Final shapes → train: {train_eng.shape}, test: {test_eng.shape}")


STEP 6: Target Aggregation Features
  ✓ geohash
  ✓ geo3
  ✓ geo4
  ✓ geo5
  ✓ hhmm
  ✓ hour
  ✓ RoadType_enc
  ✓ road_lanes
  ✓ geo4_hour
  ✓ geo4_hhmm
  ✓ geo4_day
  ✓ road_hour
  ✓ road_hhmm

  Final shapes → train: (77299, 104), test: (41778, 102)


In [8]:
# ─────────────────────────── 7. PREPARE X / y ────────────────────
DROP_COLS = ['Index','geohash','day','timestamp','demand','log_demand',
             'geo3','geo4','geo5',
             'geo4_hour','geo4_hhmm','geo4_day',
             'road_hour','road_hhmm','road_lanes',
             'RoadType','Weather']

feat_cols = [c for c in train_eng.columns
             if c not in DROP_COLS and train_eng[c].dtype != object]

X      = train_eng[feat_cols].fillna(-999)
y      = train_eng[TARGET]            # log1p(demand)
y_raw  = train_eng['demand']          # raw demand for scoring
X_test = test_eng[feat_cols].fillna(-999)

print(f"\n  Features: {len(feat_cols)}")
print(f"  X: {X.shape}   X_test: {X_test.shape}")
print("\n  Feature list:")
for i, f in enumerate(feat_cols, 1):
    print(f"    {i:02d}. {f}")


  Features: 87
  X: (77299, 87)   X_test: (41778, 87)

  Feature list:
    01. NumberofLanes
    02. LargeVehicles
    03. Landmarks
    04. Temperature
    05. hour
    06. minute
    07. hhmm
    08. lat
    09. lon
    10. is_peak_am
    11. is_peak_pm
    12. is_night
    13. is_midday
    14. hour_sin
    15. hour_cos
    16. hhmm_sin
    17. hhmm_cos
    18. day_rel
    19. is_highway
    20. is_street
    21. is_residential
    22. high_capacity
    23. temp_missing
    24. RoadType_enc
    25. Weather_enc
    26. geo3_enc
    27. geo4_enc
    28. geo5_enc
    29. geohash_enc
    30. geo4_hour_enc
    31. geo4_hhmm_enc
    32. geo4_day_enc
    33. road_hour_enc
    34. road_hhmm_enc
    35. road_lanes_enc
    36. gh_mean
    37. gh_median
    38. gh_std
    39. gh_max
    40. geo3_mean
    41. geo3_median
    42. geo3_std
    43. geo3_max
    44. geo4_mean
    45. geo4_median
    46. geo4_std
    47. geo4_max
    48. geo5_mean
    49. geo5_median
    50. geo5_std
    51. geo5_m

In [9]:
# ─────────────────────────── 8. MODEL TRAINING  ─────────────
kf = KFold(n_splits=NFOLDS, shuffle=True, random_state=SEED)

def competition_score(y_true, y_pred):
    """Competition metric on RAW (non-log) scale."""
    pred_raw = np.expm1(y_pred)
    pred_raw = np.clip(pred_raw, 0, None)
    return max(0, 100 * r2_score(y_true, pred_raw))

# ── 8A. LightGBM ───────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 8A: LightGBM")
print("=" * 60)

lgb_params = dict(
    objective         = 'regression',
    metric            = 'rmse',
    n_estimators      = 5000,
    learning_rate     = 0.01,
    num_leaves        = 255,
    max_depth         = -1,
    min_child_samples = 15,
    feature_fraction  = 0.7,
    bagging_fraction  = 0.7,
    bagging_freq      = 5,
    reg_alpha         = 0.05,
    reg_lambda        = 0.1,
    random_state      = SEED,
    n_jobs            = -1,
    verbose           = -1,
)

oof_lgb  = np.zeros(len(X))
test_lgb = np.zeros(len(X_test))
lgb_last = None

for fold, (tr_i, val_i) in enumerate(kf.split(X)):
    m = lgb.LGBMRegressor(**lgb_params)
    m.fit(
        X.iloc[tr_i], y.iloc[tr_i],
        eval_set  = [(X.iloc[val_i], y.iloc[val_i])],
        callbacks = [lgb.early_stopping(200, verbose=False),
                     lgb.log_evaluation(period=-1)]
    )
    oof_lgb[val_i]  = m.predict(X.iloc[val_i])
    test_lgb       += m.predict(X_test) / NFOLDS
    lgb_last        = m
    sc = competition_score(y_raw.iloc[val_i], oof_lgb[val_i])
    print(f"  Fold {fold+1}: {sc:.3f}  (best_iter={m.best_iteration_})")

lgb_score = competition_score(y_raw, oof_lgb)
print(f"\n  ★ LightGBM Score: {lgb_score:.3f}")

# Feature importance
fi = pd.DataFrame({'feature': feat_cols,
                   'importance': lgb_last.feature_importances_})
fi.sort_values('importance', ascending=False, inplace=True)
plt.figure(figsize=(10, 10))
sns.barplot(data=fi.head(30), x='importance', y='feature', palette='viridis')
plt.title('Top-30 Feature Importances', fontsize=13)
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150)
plt.close()
print("  feature_importance.png saved")

# ── 8B. XGBoost ────────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 8B: XGBoost")
print("=" * 60)

xgb_params = dict(
    n_estimators          = 5000,
    learning_rate         = 0.01,
    max_depth             = 8,
    min_child_weight      = 5,
    subsample             = 0.7,
    colsample_bytree      = 0.7,
    reg_alpha             = 0.05,
    reg_lambda            = 1.0,
    random_state          = SEED,
    n_jobs                = -1,
    verbosity             = 0,
    tree_method           = 'hist',
    early_stopping_rounds = 200,   # ← XGBoost 2.x: in constructor
)

oof_xgb  = np.zeros(len(X))
test_xgb = np.zeros(len(X_test))

for fold, (tr_i, val_i) in enumerate(kf.split(X)):
    xm = xgb.XGBRegressor(**xgb_params)
    xm.fit(
        X.iloc[tr_i], y.iloc[tr_i],
        eval_set = [(X.iloc[val_i], y.iloc[val_i])],
        verbose  = False,
    )
    oof_xgb[val_i]  = xm.predict(X.iloc[val_i])
    test_xgb       += xm.predict(X_test) / NFOLDS
    sc = competition_score(y_raw.iloc[val_i], oof_xgb[val_i])
    print(f"  Fold {fold+1}: {sc:.3f}  (best_iter={xm.best_iteration})")

xgb_score = competition_score(y_raw, oof_xgb)
print(f"\n  ★ XGBoost Score: {xgb_score:.3f}")

# ── 8C. CatBoost ───────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 8C: CatBoost")
print("=" * 60)

cat_params = dict(
    iterations          = 5000,
    learning_rate       = 0.01,
    depth               = 8,
    l2_leaf_reg         = 3,
    random_strength     = 1,
    bagging_temperature = 0.5,
    od_type             = 'Iter',
    od_wait             = 200,
    random_seed         = SEED,
    verbose             = False,
    task_type           = 'CPU',
)

oof_cat  = np.zeros(len(X))
test_cat = np.zeros(len(X_test))

for fold, (tr_i, val_i) in enumerate(kf.split(X)):
    cm = CatBoostRegressor(**cat_params)
    cm.fit(
        X.iloc[tr_i], y.iloc[tr_i],
        eval_set       = (X.iloc[val_i], y.iloc[val_i]),
        use_best_model = True,
    )
    oof_cat[val_i]  = cm.predict(X.iloc[val_i])
    test_cat       += cm.predict(X_test) / NFOLDS
    sc = competition_score(y_raw.iloc[val_i], oof_cat[val_i])
    print(f"  Fold {fold+1}: {sc:.3f}")

cat_score = competition_score(y_raw, oof_cat)
print(f"\n  ★ CatBoost Score: {cat_score:.3f}")


STEP 8A: LightGBM
  Fold 1: 96.182  (best_iter=3152)
  Fold 2: 96.080  (best_iter=2945)
  Fold 3: 96.246  (best_iter=2533)
  Fold 4: 95.802  (best_iter=1862)
  Fold 5: 96.080  (best_iter=2908)

  ★ LightGBM Score: 96.081
  feature_importance.png saved

STEP 8B: XGBoost
  Fold 1: 96.216  (best_iter=4985)
  Fold 2: 96.161  (best_iter=4998)
  Fold 3: 96.318  (best_iter=4999)
  Fold 4: 95.857  (best_iter=4724)
  Fold 5: 96.157  (best_iter=4999)

  ★ XGBoost Score: 96.145

STEP 8C: CatBoost
  Fold 1: 95.915
  Fold 2: 95.880
  Fold 3: 96.052
  Fold 4: 95.576
  Fold 5: 95.981

  ★ CatBoost Score: 95.884


In [10]:
# ─────────────────────────── 9. BLEND ───────────────────────
print("\n" + "=" * 60)
print("STEP 9: Weighted Blend")
print("=" * 60)

total = lgb_score + xgb_score + cat_score
w_lgb = lgb_score / total
w_xgb = xgb_score / total
w_cat = cat_score / total
print(f"  Weights → LGB:{w_lgb:.3f}  XGB:{w_xgb:.3f}  CAT:{w_cat:.3f}")

oof_blend  = w_lgb * oof_lgb  + w_xgb * oof_xgb  + w_cat * oof_cat
test_blend = w_lgb * test_lgb + w_xgb * test_xgb + w_cat * test_cat

blend_score = competition_score(y_raw, oof_blend)

print(f"\n  ┌──────────────────────────┐")
print(f"  │  LightGBM : {lgb_score:>7.3f}        │")
print(f"  │  XGBoost  : {xgb_score:>7.3f}        │")
print(f"  │  CatBoost : {cat_score:>7.3f}        │")
print(f"  │  BLEND  ★ : {blend_score:>7.3f}        │")
print(f"  └──────────────────────────┘")

# Back-transform from log space, clip to [0,1]
test_final = np.clip(np.expm1(test_blend), 0, 1)

# ── Diagnostics plot ────────────────────────────────────────────
oof_raw = np.clip(np.expm1(oof_blend), 0, None)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].scatter(y_raw, oof_raw, alpha=0.08, s=3, color='steelblue')
mn, mx = float(y_raw.min()), float(y_raw.max())
axes[0].plot([mn,mx],[mn,mx],'r--',lw=1.5,label='Perfect')
axes[0].set_title(f'OOF: Actual vs Predicted  (score={blend_score:.2f})')
axes[0].set_xlabel('Actual demand')
axes[0].set_ylabel('Predicted demand')
axes[0].legend()
residuals = y_raw - oof_raw
axes[1].hist(residuals, bins=80, color='coral', edgecolor='white')
axes[1].axvline(0,color='black',lw=1.5,ls='--')
axes[1].set_title('Residual Distribution')
plt.tight_layout()
plt.savefig('oof_diagnostics.png', dpi=150)
plt.close()


STEP 9: Weighted Blend
  Weights → LGB:0.333  XGB:0.334  CAT:0.333

  ┌──────────────────────────┐
  │  LightGBM :  96.081        │
  │  XGBoost  :  96.145        │
  │  CatBoost :  95.884        │
  │  BLEND  ★ :  96.147        │
  └──────────────────────────┘


In [11]:
total = lgb_score + xgb_score + cat_score

w_lgb = lgb_score / total
w_xgb = xgb_score / total
w_cat = cat_score / total

test_blend = (
    w_lgb * test_lgb +
    w_xgb * test_xgb +
    w_cat * test_cat
)

test_final = np.clip(np.expm1(test_blend), 0, 1)

submission = pd.DataFrame({
    "Index": test["Index"],
    "demand": test_final
})

submission.to_csv("submission.csv", index=False)

print("submission.csv generated")

submission.csv generated
